# ClauseXplain: Indian-set evaluation v2 (saves per-clause predictions)
Attach both datasets: `clausexplain-indian-eval` and `scan-vs-span`. Set FOLDS to `[0]` if it runs too slowly.

In [ ]:
import os, shutil, base64, subprocess, sys
BASE = "/kaggle/input/datasets/riyasuryawanshi"
EVAL_SRC, MODEL_SRC = f"{BASE}/clausexplain-indian-eval", f"{BASE}/scan-vs-span"
WORK = "/kaggle/working/clausexplain"
FOLDS = [0, 1, 2]
if os.path.exists(WORK): shutil.rmtree(WORK)
shutil.copytree(EVAL_SRC, WORK)
for name, b in {"eval_indian.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIgpldmFsX2luZGlhbi5weSAtLSBob3cgZG8gdGhlIENVQUQtdHJhaW5lZCBtb2RlbHMgYmVoYXZlIG9uIEluZGlhbi1jb250ZXh0IGNsYXVzZXM/CgogICAgcHl0aG9uIGV2YWxfaW5kaWFuLnB5IC0tcnVuX2RpciAva2FnZ2xlL3dvcmtpbmcvc3RhZ2VCIC0tZm9sZCAwIFwKICAgICAgICAtLWNrcHQgY2twdF9qb2ludF9jbHNfc3Bhbl9mb2xkMC5wdCBcCiAgICAgICAgLS1pbmRpYW5fZGlyIGluZGlhbl9zZXQvYnVpbHQgXAogICAgICAgIC0tbWFwcGVkX2pzb25sIGNsYXVzZXNfbWFwcGVkLmpzb25sIC0tY3VhZF9qc29uIENVQURfdjEuanNvbiBcCiAgICAgICAgLS1ydWxlcyBpbmRpYW5fbGVnYWxfcnVsZXMuY3N2CgooYnVpbGQgdGhlIEluZGlhbiBzZXQgZmlyc3Qgd2l0aCBidWlsZF9pbmRpYW5fdGVzdHNldC5weSkKCldIQVQgSVQgUkVQT1JUUwogICAgMS4gQ2F0ZWdvcnkgY2xhc3NpZmljYXRpb24gb24gSW5kaWFuIGNsYXVzZXMgd2hvc2UgY2F0ZWdvcnkgdGhlIG1vZGVsIHdhcyB0cmFpbmVkCiAgICAgICBvbiAoMTIgb2YgMTgpLCB3aXRoIGEgV2lsc29uIDk1JSBpbnRlcnZhbCwgbmV4dCB0byB0aGUgVEYtSURGIGJhc2VsaW5lIHRyYWluZWQKICAgICAgIG9uIHRoZSBzYW1lIENVQUQgdHJhaW5pbmcgc3BsaXQuCiAgICAyLiBTcGFuIGxvY2FsaXNhdGlvbiBvbiB0aG9zZSBjbGF1c2VzICh3aW5kb3dlZCwgc2FtZSBzZXR0aW5ncyBhcyB0cmFpbmluZyksCiAgICAgICBuZXh0IHRvIHRoZSB3aG9sZS13aW5kb3cgYW5kIGxleGljYWwtY2VudHJvaWQgYmFzZWxpbmVzLgogICAgMy4gV2hhdCB0aGUgbW9kZWwgcHJlZGljdHMgZm9yIGNsYXVzZXMgaW4gdGhlIDYgY2F0ZWdvcmllcyBpdCBoYXMgTkVWRVIgc2VlbgogICAgICAgKENvbmZpZGVudGlhbGl0eSwgRm9yY2UgTWFqZXVyZSwgSW5kZW1uaWZpY2F0aW9uLCBNb2RpZmljYXRpb24sIFBheW1lbnQsCiAgICAgICBQcml2YWN5KS4gIEl0IGNhbm5vdCBiZSByaWdodCBvbiB0aGVzZTsgdGhlIHRhYmxlIHNob3dzIHdoYXQgaXQgZG9lcyBpbnN0ZWFkLgogICAgNC4gUnVsZS1sYXllciBmbGFncyBvbiBldmVyeSBJbmRpYW4gY2xhdXNlIChpbmRpYW5fcnVsZV9mbGFncy5jc3YsIGZvciByZXZpZXcpLgoKSE9ORVNUIExJTUlUUwogICAgQSB0ZXN0IHNldCBvZiBhIGZldyBkb3plbiBjbGF1c2VzIGdpdmVzIHdpZGUgaW50ZXJ2YWxzLCBhbmQgYWxsIG51bWJlcnMgY29tZSBmcm9tCiAgICBPTkUgY2hlY2twb2ludC4gIERpZmZlcmVuY2VzIG9mIGEgZmV3IHBvaW50cyBhcmUgbm9pc2UuICBUaGUgcG9pbnQgb2YgdGhpcyB0ZXN0IGlzCiAgICB0aGUgc2l6ZSBvZiB0aGUgZ2FwIHRvIHRoZSBDVUFEIG51bWJlcnMgKGRvbWFpbiBzaGlmdCksIG5vdCBhIGxlYWRlcmJvYXJkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgU3Vic2V0CgppbXBvcnQgbWV0cmljcyBhcyBNCmZyb20gYnVpbGRfaW5kaWFuX3Rlc3RzZXQgaW1wb3J0IE5PX1RSQUlOSU5HX0RBVEEKZnJvbSBkYXRhc2V0IGltcG9ydCAoQ0FURUdPUklFUywgQ2xhdXNlRGF0YXNldCwgYXR0YWNoX2NvbnRyYWN0X3RleHQsIGxvYWRfbWFwcGVkLCBtYWtlX2NvbGxhdG9yKQpmcm9tIHJ1bGVzX2VuZ2luZSBpbXBvcnQgYXBwbHlfcnVsZXMsIGxvYWRfcnVsZXMKZnJvbSBzY2FuX3ZzX3NwYW4gaW1wb3J0IGxvYWRfbW9kZWwKZnJvbSB0cmFpbl9waGFzZTMgaW1wb3J0IGNob29zZV9hbXBfZHR5cGUsIGNsYXNzaWZpY2F0aW9uX2Jhc2VsaW5lcywgZXZhbHVhdGUsIHNwYW5fYmFzZWxpbmVzCgoKZGVmIHdpbHNvbihrOiBpbnQsIG46IGludCwgejogZmxvYXQgPSAxLjk2KToKICAgIGlmIG4gPT0gMDoKICAgICAgICByZXR1cm4gKGZsb2F0KCJuYW4iKSwgZmxvYXQoIm5hbiIpKQogICAgcCA9IGsgLyBuCiAgICBkID0gMSArIHogKiB6IC8gbgogICAgYyA9IChwICsgeiAqIHogLyAoMiAqIG4pKSAvIGQKICAgIGggPSB6ICogbWF0aC5zcXJ0KHAgKiAoMSAtIHApIC8gbiArIHogKiB6IC8gKDQgKiBuICogbikpIC8gZAogICAgcmV0dXJuIChtYXgoMC4wLCBjIC0gaCksIG1pbigxLjAsIGMgKyBoKSkKCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGRpY3Q6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcnVuX2RpciIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZm9sZCIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2twdCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9ImNoZWNrcG9pbnQgdXNlZCBmb3IgY2F0ZWdvcnkgQU5EIHNwYW4gKGUuZy4gdGhlIGpvaW50IG1vZGVsKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2twdF9zcGFuIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJvcHRpb25hbCBkaWZmZXJlbnQgY2hlY2twb2ludCBmb3IgdGhlIHNwYW4gaGVhZCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taW5kaWFuX2RpciIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWFwcGVkX2pzb25sIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJDVUFEIGNsYXVzZXNfbWFwcGVkLmpzb25sIChmb3IgYmFzZWxpbmVzKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY3VhZF9qc29uIiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJ1bGVzIiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNsYXVzZV9iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNwYW5fYmF0Y2giLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRpbnkiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoYXJndikKCiAgICBydW4sIGluZCA9IFBhdGgoYS5ydW5fZGlyKSwgUGF0aChhLmluZGlhbl9kaXIpCiAgICBjZmcgPSBqc29uLmxvYWRzKChydW4gLyAicGFydGlhbCIgLyAiY29uZmlnLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNmZy5zZXRkZWZhdWx0KCJtYXhfbGVuX2NsYXVzZSIsIDUxMikKICAgIGlmIGEudGlueToKICAgICAgICBjZmdbInRpbnkiXSA9IFRydWUKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgYW1wX2R0eXBlID0gY2hvb3NlX2FtcF9kdHlwZShkZXZpY2UsICJub25lIiBpZiBhLnRpbnkgZWxzZSAiYXV0byIpCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgogICAgdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoc3RyKHJ1biAvICJ0aW55X3Rva2VuaXplciIpIGlmIGEudGlueSBlbHNlIGNmZ1siZW5jb2RlciJdKQogICAgcGFkX211bHQgPSAzMiBpZiBhLnRpbnkgZWxzZSA1MTIKCiAgICAjIC0tLS0gSW5kaWFuIGRhdGEgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBleF9tYXAgPSBsb2FkX21hcHBlZChpbmQgLyAiaW5kaWFuX21hcHBlZC5qc29ubCIpCiAgICB0ZXh0cyA9IGpzb24ubG9hZHMoKGluZCAvICJpbmRpYW5fY29udHJhY3RzLmpzb24iKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBhdHRhY2hfY29udHJhY3RfdGV4dChleF9tYXAsIHRleHRzLCBzdHJpY3Q9VHJ1ZSkKICAgIGV2ZXJ5dGhpbmcgPSBsaXN0KGV4X21hcC52YWx1ZXMoKSkKICAgIHNlZW4gPSBbZSBmb3IgZSBpbiBldmVyeXRoaW5nIGlmIGUuY2F0ZWdvcnkgbm90IGluIE5PX1RSQUlOSU5HX0RBVEFdCiAgICB1bnNlZW4gPSBbZSBmb3IgZSBpbiBldmVyeXRoaW5nIGlmIGUuY2F0ZWdvcnkgaW4gTk9fVFJBSU5JTkdfREFUQV0KICAgIHByaW50KGYie2xlbihldmVyeXRoaW5nKX0gSW5kaWFuIGNsYXVzZXM6IHtsZW4oc2Vlbil9IGluIGNhdGVnb3JpZXMgdGhlIG1vZGVsIHdhcyB0cmFpbmVkIG9uLCAiCiAgICAgICAgICBmIntsZW4odW5zZWVuKX0gaW4gY2F0ZWdvcmllcyBpdCBuZXZlciBzYXciKQoKICAgICMgLS0tLSBDVUFEIHRyYWluaW5nIHNwbGl0IGZvciBiYXNlbGluZXMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgdHJhaW5fZXggPSBOb25lCiAgICBpZiBhLm1hcHBlZF9qc29ubCBhbmQgYS5jdWFkX2pzb246CiAgICAgICAgZnJvbSBkYXRhc2V0IGltcG9ydCBsb2FkX2NvbnRyYWN0X3RleHRzCiAgICAgICAgY21hcCA9IGxvYWRfbWFwcGVkKGEubWFwcGVkX2pzb25sKQogICAgICAgIGF0dGFjaF9jb250cmFjdF90ZXh0KGNtYXAsIGxvYWRfY29udHJhY3RfdGV4dHMoYS5jdWFkX2pzb24pLCBzdHJpY3Q9VHJ1ZSkKICAgICAgICBwb29sID0gW2UgZm9yIGUgaW4gY21hcC52YWx1ZXMoKSBpZiBlLmhhc19jbHNdCiAgICAgICAgaWYgY2ZnLmdldCgibWF4X2NvbnRyYWN0cyIpOgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJiYXNlbGluZXMgbmVlZCB0aGUgZnVsbCBDVUFEIHBvb2w7IHJlcnVuIHdpdGhvdXQgLS1tYXhfY29udHJhY3RzIHJ1bnMiKQogICAgICAgIGMyaSA9IHtlLmNsYXVzZV9pZDogaSBmb3IgaSwgZSBpbiBlbnVtZXJhdGUocG9vbCl9CiAgICAgICAgc3AgPSBydW4gLyAicGFydGlhbCIgLyAic3BsaXRzLmpzb24iCiAgICAgICAgaWYgc3AuZXhpc3RzKCk6CiAgICAgICAgICAgIHRyYWluX2V4ID0gW3Bvb2xbYzJpW2NdXSBmb3IgYyBpbiBqc29uLmxvYWRzKHNwLnJlYWRfdGV4dCgpKVthLmZvbGRdWzBdXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KCJXQVJOSU5HOiBzcGxpdHMuanNvbiBtaXNzaW5nOyBiYXNlbGluZXMgc2tpcHBlZCIpCgogICAgIyAtLS0tIG1vZGVsIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBtX2NscyA9IGxvYWRfbW9kZWwoYS5ja3B0LCBjZmcsIHRvaywgZGV2aWNlKQogICAgbV9zcGFuID0gbV9jbHMgaWYgbm90IGEuY2twdF9zcGFuIGVsc2UgbG9hZF9tb2RlbChhLmNrcHRfc3BhbiwgY2ZnLCB0b2ssIGRldmljZSkKICAgIGNvbGxhdGUgPSBtYWtlX2NvbGxhdG9yKHRvay5wYWRfdG9rZW5faWQsIHBhZF90b19tdWx0aXBsZV9vZj1wYWRfbXVsdCkKCiAgICBkZWYgbG9hZGVyc19mb3IoZXhhbXBsZXMsIHZpZXdzKToKICAgICAgICBkcyA9IENsYXVzZURhdGFzZXQoZXhhbXBsZXMsIHRvaywgbWF4X2xlbl9jbGF1c2U9Y2ZnWyJtYXhfbGVuX2NsYXVzZSJdLCBtYXhfbGVuX3NwYW49Y2ZnWyJtYXhfbGVuX3NwYW4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Bhbl9tb2RlPSJjdWFkX2NvbnRleHQiLCB3aW5kb3dfY2hhcnM9Y2ZnWyJ3aW5kb3dfY2hhcnMiXSwgc3Bhbl9wb29sPWV4YW1wbGVzLAogICAgICAgICAgICAgICAgICAgICAgICAgICB2aWV3cz12aWV3cywgdXNlX2Nscz1UcnVlLCB1c2Vfcmlzaz1GYWxzZSwgc2VlZD1jZmdbInNlZWQiXSArIGEuZm9sZCkKICAgICAgICBvdXQgPSB7fQogICAgICAgIGZvciB2IGluIHZpZXdzOgogICAgICAgICAgICBpZHMgPSBbaSBmb3IgaSwgKHZ2LCBfKSBpbiBlbnVtZXJhdGUoZHMuaXRlbXMpIGlmIHZ2ID09IHZdCiAgICAgICAgICAgIGlmIGlkczoKICAgICAgICAgICAgICAgIG91dFt2XSA9IERhdGFMb2FkZXIoU3Vic2V0KGRzLCBpZHMpLCBiYXRjaF9zaXplPWEuY2xhdXNlX2JhdGNoIGlmIHYgPT0gImNsYXVzZSIgZWxzZSBhLnNwYW5fYmF0Y2gsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNodWZmbGU9RmFsc2UsIGNvbGxhdGVfZm49Y29sbGF0ZSkKICAgICAgICByZXR1cm4gZHMsIG91dAoKICAgIHJlczogZGljdCA9IHsibl90b3RhbCI6IGxlbihldmVyeXRoaW5nKSwgIm5fc2VlbiI6IGxlbihzZWVuKSwgIm5fdW5zZWVuIjogbGVuKHVuc2Vlbil9CiAgICB0cmFpbl9jaGFycyA9IChbZS5jaGFyX2VuZCAtIGUuY2hhcl9zdGFydCBmb3IgZSBpbiB0cmFpbl9leF0gaWYgdHJhaW5fZXgKICAgICAgICAgICAgICAgICAgIGVsc2UgW2UuY2hhcl9lbmQgLSBlLmNoYXJfc3RhcnQgZm9yIGUgaW4gc2Vlbl0pCiAgICBtZWQgPSBmbG9hdChucC5tZWRpYW4odHJhaW5fY2hhcnMpKSBpZiB0cmFpbl9jaGFycyBlbHNlIE5vbmUKCiAgICBpZiBzZWVuOgogICAgICAgIF8sIGxvID0gbG9hZGVyc19mb3Ioc2VlbiwgKCJjbGF1c2UiLCAic3BhbiIpKQogICAgICAgIHIxID0gZXZhbHVhdGUobV9jbHMsIHtrOiB2IGZvciBrLCB2IGluIGxvLml0ZW1zKCkgaWYgayA9PSAiY2xhdXNlIn0sICgiY2xzIiwpLCBkZXZpY2UsIGFtcF9kdHlwZSwgbWVkKQogICAgICAgIHIyID0gZXZhbHVhdGUobV9zcGFuLCB7azogdiBmb3IgaywgdiBpbiBsby5pdGVtcygpIGlmIGsgPT0gInNwYW4ifSwgKCJzcGFuIiwpLCBkZXZpY2UsIGFtcF9kdHlwZSwgbWVkKQogICAgICAgIGNscyA9IHIxLmdldCgiY2xzIikKICAgICAgICByZXNbImNscyJdID0gY2xzCiAgICAgICAgayA9IGludChyb3VuZChjbHNbImFjY3VyYWN5Il0gKiBjbHNbIm4iXSkpIGlmIGNscyBlbHNlIDAKICAgICAgICByZXNbImNsc19hY2NfY2k5NSJdID0gd2lsc29uKGssIGNsc1sibiJdKSBpZiBjbHMgZWxzZSBOb25lCiAgICAgICAgcmVzWyJzcGFuIl0gPSByMi5nZXQoInNwYW4iKQogICAgICAgIHJlc1sic3Bhbl9vd25fbGFiZWwiXSA9IHIyLmdldCgic3Bhbl9vd25fbGFiZWwiKQogICAgICAgIHJlc1sic3Bhbl9yYW5kb21fYmFzZWxpbmUiXSA9IHIyLmdldCgic3Bhbl9yYW5kb21fYmFzZWxpbmUiKQogICAgICAgIGlmIHRyYWluX2V4OgogICAgICAgICAgICByZXNbImJhc2VsaW5lc19jbHMiXSA9IGNsYXNzaWZpY2F0aW9uX2Jhc2VsaW5lcyh0cmFpbl9leCwgc2VlbikKICAgICAgICAgICAgdGIgPSByZXNbImJhc2VsaW5lc19jbHMiXS5nZXQoInRmaWRmX2xvZ3JlZyIpCiAgICAgICAgICAgIGlmIHRiOgogICAgICAgICAgICAgICAga3QgPSBpbnQocm91bmQodGJbImFjY3VyYWN5Il0gKiB0YlsibiJdKSkKICAgICAgICAgICAgICAgIHJlc1sidGZpZGZfYWNjX2NpOTUiXSA9IHdpbHNvbihrdCwgdGJbIm4iXSkKICAgICAgICAgICAgXywgbG8yID0gbG9hZGVyc19mb3Ioc2VlbiwgKCJzcGFuIiwpKQogICAgICAgICAgICByZXNbImJhc2VsaW5lc19zcGFuIl0gPSBzcGFuX2Jhc2VsaW5lcyh0cmFpbl9leCwgbG8yWyJzcGFuIl0pIGlmICJzcGFuIiBpbiBsbzIgZWxzZSB7fQoKICAgICMgLS0tLSB3aGF0IHRoZSBtb2RlbCBkb2VzIG9uIGNhdGVnb3JpZXMgaXQgbmV2ZXIgc2F3IC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGNvbmZ1c2lvbiA9IGRlZmF1bHRkaWN0KENvdW50ZXIpCiAgICBpZiB1bnNlZW46CiAgICAgICAgXywgbHUgPSBsb2FkZXJzX2Zvcih1bnNlZW4sICgiY2xhdXNlIiwpKQogICAgICAgIG1fY2xzLmV2YWwoKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBmb3IgYmF0Y2ggaW4gbHUuZ2V0KCJjbGF1c2UiLCBbXSk6CiAgICAgICAgICAgICAgICBiID0ge2s6IHYudG8oZGV2aWNlKSBmb3IgaywgdiBpbiBiYXRjaC5pdGVtcygpIGlmIGsgIT0gIm1ldGEifQogICAgICAgICAgICAgICAgbGFiID0gYmF0Y2hbImNsc19sYWJlbHMiXS50b2xpc3QoKQogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdChkZXZpY2VfdHlwZT1kZXZpY2UudHlwZSwgZHR5cGU9YW1wX2R0eXBlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmFibGVkPWRldmljZS50eXBlID09ICJjdWRhIiBhbmQgYW1wX2R0eXBlIGlzIG5vdCBOb25lKToKICAgICAgICAgICAgICAgICAgICBwcmVkID0gbV9jbHMoKipiKVsiY2xzX2xvZ2l0cyJdLmFyZ21heCgtMSkudG9saXN0KCkKICAgICAgICAgICAgICAgIGZvciB0LCBwIGluIHppcChsYWIsIHByZWQpOgogICAgICAgICAgICAgICAgICAgIGNvbmZ1c2lvbltDQVRFR09SSUVTW3RdXVtDQVRFR09SSUVTW3BdXSArPSAxCiAgICByZXNbInVuc2Vlbl9jYXRlZ29yeV9wcmVkaWN0aW9ucyJdID0ge2s6IGRpY3QodikgZm9yIGssIHYgaW4gY29uZnVzaW9uLml0ZW1zKCl9CgogICAgIyAtLS0tIHBlci1jbGF1c2UgcHJlZGljdGlvbnMgKGFsbCBjbGF1c2VzKSAtPiBpbmRpYW5fcHJlZGljdGlvbnMuY3N2IC0tLS0tLS0tLS0tLS0tLQogICAgIyBMZXRzIHlvdSBzbGljZSByZXN1bHRzIGJ5IGRvY3VtZW50LCBkcm9wIG5vaXN5IGRvY3VtZW50cywgb3IgaW5zcGVjdCBlcnJvcnMuCiAgICBfLCBsYWxsID0gbG9hZGVyc19mb3IoZXZlcnl0aGluZywgKCJjbGF1c2UiLCkpCiAgICBwcmVkX3Jvd3MgPSBbXQogICAgbV9jbHMuZXZhbCgpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgYmF0Y2ggaW4gbGFsbC5nZXQoImNsYXVzZSIsIFtdKToKICAgICAgICAgICAgYiA9IHtrOiB2LnRvKGRldmljZSkgZm9yIGssIHYgaW4gYmF0Y2guaXRlbXMoKSBpZiBrICE9ICJtZXRhIn0KICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdChkZXZpY2VfdHlwZT1kZXZpY2UudHlwZSwgZHR5cGU9YW1wX2R0eXBlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVuYWJsZWQ9ZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGFuZCBhbXBfZHR5cGUgaXMgbm90IE5vbmUpOgogICAgICAgICAgICAgICAgbG9naXRzID0gbV9jbHMoKipiKVsiY2xzX2xvZ2l0cyJdLmZsb2F0KCkKICAgICAgICAgICAgcHJvYiA9IHRvcmNoLnNvZnRtYXgobG9naXRzLCAtMSkKICAgICAgICAgICAgdG9wID0gcHJvYi5hcmdtYXgoLTEpLnRvbGlzdCgpCiAgICAgICAgICAgIGNvbmYgPSBwcm9iLm1heCgtMSkudmFsdWVzLnRvbGlzdCgpCiAgICAgICAgICAgIGZvciBtZXRhLCB0LCBwXywgYyBpbiB6aXAoYmF0Y2hbIm1ldGEiXSwgYmF0Y2hbImNsc19sYWJlbHMiXS50b2xpc3QoKSwgdG9wLCBjb25mKToKICAgICAgICAgICAgICAgIGNpZCA9IG1ldGFbImNsYXVzZV9pZCJdCiAgICAgICAgICAgICAgICBwcmVkX3Jvd3MuYXBwZW5kKFtjaWQsIGNpZC5zcGxpdCgiX18iKVswXSwgQ0FURUdPUklFU1t0XSwgQ0FURUdPUklFU1twX10sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbnQoQ0FURUdPUklFU1t0XSBub3QgaW4gTk9fVFJBSU5JTkdfREFUQSksIGludCh0ID09IHBfKSwgZiJ7YzouM2Z9IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV4X21hcFtjaWRdLnRleHRbOjMwMF0ucmVwbGFjZSgiXG4iLCAiICIpXSkKICAgIHdpdGggb3BlbihpbmQgLyAiaW5kaWFuX3ByZWRpY3Rpb25zLmNzdiIsICJ3IiwgbmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgdyA9IGNzdi53cml0ZXIoZmgpCiAgICAgICAgdy53cml0ZXJvdyhbImNsYXVzZV9pZCIsICJkb2N1bWVudCIsICJ0cnVlX2NhdGVnb3J5IiwgInByZWRfY2F0ZWdvcnkiLCAic2Vlbl9jYXRlZ29yeSIsCiAgICAgICAgICAgICAgICAgICAgImNvcnJlY3QiLCAiY29uZmlkZW5jZSIsICJjbGF1c2VfdGV4dCJdKQogICAgICAgIHcud3JpdGVyb3dzKHByZWRfcm93cykKICAgIHByaW50KGYid3JvdGUge2xlbihwcmVkX3Jvd3MpfSBwZXItY2xhdXNlIHByZWRpY3Rpb25zIHRvIHtpbmQgLyAnaW5kaWFuX3ByZWRpY3Rpb25zLmNzdid9IikKCiAgICAjIC0tLS0gcnVsZSBmbGFncyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGlmIGEucnVsZXM6CiAgICAgICAgcnVsZXMgPSBsb2FkX3J1bGVzKGEucnVsZXMpCiAgICAgICAgcm93cywgdGFsbHkgPSBbXSwgQ291bnRlcigpCiAgICAgICAgZm9yIGUgaW4gZXZlcnl0aGluZzoKICAgICAgICAgICAgZm9yIGYgaW4gYXBwbHlfcnVsZXMoZS50ZXh0LCBlLmNhdGVnb3J5LCBydWxlcyk6CiAgICAgICAgICAgICAgICB0YWxseVsoZi5ydWxlX2lkLCBmLnN0YXR1cyldICs9IDEKICAgICAgICAgICAgICAgIGlmIGYuc3RhdHVzICE9ICJub25lIjoKICAgICAgICAgICAgICAgICAgICBldiA9ICI7ICIuam9pbih0IGZvciBfLCBfLCB0IGluIGYuZXZpZGVuY2VbOjRdKQogICAgICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKFtlLmNsYXVzZV9pZCwgZS5jYXRlZ29yeSwgZi5ydWxlX2lkLCBmLnN0YXR1cywgIiB8ICIuam9pbihmLnJlYXNvbnMpLCBldiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZi5ub3RlLCBlLnRleHRbOjMwMF1dKQogICAgICAgIHdpdGggb3BlbihpbmQgLyAiaW5kaWFuX3J1bGVfZmxhZ3MuY3N2IiwgInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmaDoKICAgICAgICAgICAgdyA9IGNzdi53cml0ZXIoZmgpCiAgICAgICAgICAgIHcud3JpdGVyb3coWyJjbGF1c2VfaWQiLCAiY2F0ZWdvcnkiLCAicnVsZV9pZCIsICJzdGF0dXMiLCAicmVhc29ucyIsICJldmlkZW5jZSIsICJub3RlIiwgImNsYXVzZV90ZXh0Il0pCiAgICAgICAgICAgIHcud3JpdGVyb3dzKHJvd3MpCiAgICAgICAgcmVzWyJydWxlX3N0YXR1c19jb3VudHMiXSA9IHtmIntyfTp7c30iOiBuIGZvciAociwgcyksIG4gaW4gc29ydGVkKHRhbGx5Lml0ZW1zKCkpfQoKICAgICMgLS0tLSByZWZlcmVuY2UgbnVtYmVycyBmcm9tIHRoZSBDVUFEIHJ1biAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgcmogPSBydW4gLyAicmVzdWx0cy5qc29uIgogICAgaWYgcmouZXhpc3RzKCk6CiAgICAgICAgYWdnID0ganNvbi5sb2Fkcyhyai5yZWFkX3RleHQoKSkuZ2V0KCJhZ2dyZWdhdGUiLCB7fSkKICAgICAgICByZXNbImN1YWRfcmVmZXJlbmNlIl0gPSB7azogeyJjbHNfbWFjcm9fZjEiOiAodi5nZXQoImNscyIpIG9yIHt9KS5nZXQoIm1lYW4iLCB7fSkuZ2V0KCJtYWNyb19mMSIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNsc19taWNyb19mMSI6ICh2LmdldCgiY2xzIikgb3Ige30pLmdldCgibWVhbiIsIHt9KS5nZXQoIm1pY3JvX2YxIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic3Bhbl90b2tlbl9mMSI6ICh2LmdldCgic3BhbiIpIG9yIHt9KS5nZXQoIm1lYW4iLCB7fSkuZ2V0KCJ0b2tlbl9mMSIpfQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgaywgdiBpbiBhZ2cuaXRlbXMoKX0KCiAgICAoaW5kIC8gImluZGlhbl9ldmFsLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzLCBpbmRlbnQ9MSwgZGVmYXVsdD1mbG9hdCksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgIyAtLS0tIG1hcmtkb3duIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZjMgPSBsYW1iZGEgeDogIm4vYSIgaWYgeCBpcyBOb25lIG9yIChpc2luc3RhbmNlKHgsIGZsb2F0KSBhbmQgbWF0aC5pc25hbih4KSkgZWxzZSBmInt4Oi4zZn0iCiAgICBMID0gWyIjIEluZGlhbi1jb250ZXh0IGV2YWx1YXRpb24iLCAiIiwKICAgICAgICAgZiJ7cmVzWyduX3RvdGFsJ119IGNsYXVzZXM6IHtyZXNbJ25fc2VlbiddfSBpbiBjYXRlZ29yaWVzIHNlZW4gaW4gdHJhaW5pbmcsIHtyZXNbJ25fdW5zZWVuJ119IGluIHVuc2VlbiBjYXRlZ29yaWVzLiAiCiAgICAgICAgICJTbWFsbCBzYW1wbGU6IHJlYWQgZ2Fwcywgbm90IGRlY2ltYWxzLiIsICIiXQogICAgaWYgcmVzLmdldCgiY2xzIik6CiAgICAgICAgYyA9IHJlc1siY2xzIl0KICAgICAgICBMICs9IFsiIyMgQ2F0ZWdvcnkgY2xhc3NpZmljYXRpb24gKHNlZW4gY2F0ZWdvcmllcyBvbmx5KSIsICIiLAogICAgICAgICAgICAgICJ8IE1vZGVsIHwgbiB8IGFjY3VyYWN5IHwgOTUlIGludGVydmFsIHwgbWFjcm8tRjEgfCIsICJ8LS0tfC0tLXwtLS18LS0tfC0tLXwiLAogICAgICAgICAgICAgIGYifCBDbGF1c2VYcGxhaW4gY2hlY2twb2ludCB8IHtjWyduJ119IHwge2YzKGNbJ2FjY3VyYWN5J10pfSB8ICIKICAgICAgICAgICAgICBmIntmMyhyZXNbJ2Nsc19hY2NfY2k5NSddWzBdKX0gdG8ge2YzKHJlc1snY2xzX2FjY19jaTk1J11bMV0pfSB8IHtmMyhjWydtYWNyb19mMSddKX0gfCJdCiAgICAgICAgZm9yIG4sIGIgaW4gcmVzLmdldCgiYmFzZWxpbmVzX2NscyIsIHt9KS5pdGVtcygpOgogICAgICAgICAgICBjaSA9IHJlcy5nZXQoInRmaWRmX2FjY19jaTk1IikgaWYgbiA9PSAidGZpZGZfbG9ncmVnIiBlbHNlIE5vbmUKICAgICAgICAgICAgTC5hcHBlbmQoZiJ8IHtufSAodHJhaW5lZCBvbiBDVUFEIGZvbGQge2EuZm9sZH0pIHwge2JbJ24nXX0gfCB7ZjMoYlsnYWNjdXJhY3knXSl9IHwgIgogICAgICAgICAgICAgICAgICAgICArIChmIntmMyhjaVswXSl9IHRvIHtmMyhjaVsxXSl9IiBpZiBjaSBlbHNlICIiKSArIGYiIHwge2YzKGJbJ21hY3JvX2YxJ10pfSB8IikKICAgICAgICBMLmFwcGVuZCgiIikKICAgIGlmIHJlcy5nZXQoInNwYW4iKToKICAgICAgICBMICs9IFsiIyMgU3BhbiBsb2NhbGlzYXRpb24gKHdpbmRvd2VkKSIsICIiLCAifCBNZXRob2QgfCBuIHwgdG9rZW4tRjEgfCBleGFjdCBtYXRjaCB8IElvVSB8IiwgInwtLS18LS0tfC0tLXwtLS18LS0tfCJdCiAgICAgICAgZm9yIG5hbWUsIG0gaW4gWygic3BhbiBoZWFkIChiZXN0LW1hdGNoIGdvbGQpIiwgcmVzWyJzcGFuIl0pLCAoInNwYW4gaGVhZCAob3duIGxhYmVsKSIsIHJlcy5nZXQoInNwYW5fb3duX2xhYmVsIikpLAogICAgICAgICAgICAgICAgICAgICAgICAoInJhbmRvbSBzcGFuIiwgcmVzLmdldCgic3Bhbl9yYW5kb21fYmFzZWxpbmUiKSldICsgbGlzdChyZXMuZ2V0KCJiYXNlbGluZXNfc3BhbiIsIHt9KS5pdGVtcygpKToKICAgICAgICAgICAgaWYgbToKICAgICAgICAgICAgICAgIEwuYXBwZW5kKGYifCB7bmFtZX0gfCB7aW50KG1bJ24nXSl9IHwge2YzKG1bJ3Rva2VuX2YxJ10pfSB8IHtmMyhtWydleGFjdF9tYXRjaCddKX0gfCB7ZjMobVsnaW91J10pfSB8IikKICAgICAgICBMLmFwcGVuZCgiIikKICAgIGlmIGNvbmZ1c2lvbjoKICAgICAgICBMICs9IFsiIyMgQ2F0ZWdvcmllcyB0aGUgbW9kZWwgbmV2ZXIgc2F3OiB3aGF0IGl0IHByZWRpY3RlZCBpbnN0ZWFkIiwgIiIsICJ8IFRydWUgY2F0ZWdvcnkgfCBwcmVkaWN0aW9ucyB8IiwgInwtLS18LS0tfCJdCiAgICAgICAgZm9yIHQsIGN0ciBpbiBzb3J0ZWQoY29uZnVzaW9uLml0ZW1zKCkpOgogICAgICAgICAgICBMLmFwcGVuZChmInwge3R9IHwgIiArICIsICIuam9pbihmIntwfSB4e259IiBmb3IgcCwgbiBpbiBjdHIubW9zdF9jb21tb24oKSkgKyAiIHwiKQogICAgICAgIEwuYXBwZW5kKCIiKQogICAgaWYgcmVzLmdldCgicnVsZV9zdGF0dXNfY291bnRzIik6CiAgICAgICAgTCArPSBbIiMjIFJ1bGUtbGF5ZXIgZmluZGluZ3MgKHNlZSBpbmRpYW5fcnVsZV9mbGFncy5jc3YpIiwgIiJdICsgXAogICAgICAgICAgICAgW2YiLSB7a306IHt2fSIgZm9yIGssIHYgaW4gcmVzWyJydWxlX3N0YXR1c19jb3VudHMiXS5pdGVtcygpXSArIFsiIl0KICAgIGlmIHJlcy5nZXQoImN1YWRfcmVmZXJlbmNlIik6CiAgICAgICAgTCArPSBbIiMjIFJlZmVyZW5jZTogQ1VBRCBjcm9zcy12YWxpZGF0ZWQgdGVzdCBudW1iZXJzIChtZWFuIG92ZXIgZm9sZHMpIiwgIiIsCiAgICAgICAgICAgICAgInwgTW9kZWwgfCBtYWNyby1GMSB8IG1pY3JvLUYxIHwgc3BhbiB0b2tlbi1GMSB8IiwgInwtLS18LS0tfC0tLXwtLS18Il0KICAgICAgICBmb3IgaywgdiBpbiByZXNbImN1YWRfcmVmZXJlbmNlIl0uaXRlbXMoKToKICAgICAgICAgICAgTC5hcHBlbmQoZiJ8IHtrfSB8IHtmMyh2WydjbHNfbWFjcm9fZjEnXSl9IHwge2YzKHZbJ2Nsc19taWNyb19mMSddKX0gfCB7ZjModlsnc3Bhbl90b2tlbl9mMSddKX0gfCIpCiAgICAoaW5kIC8gImluZGlhbl9ldmFsLm1kIikud3JpdGVfdGV4dCgiXG4iLmpvaW4oTCksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludCgiXG4iLmpvaW4oTCkpCiAgICByZXR1cm4gcmVzCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "summarise_predictions.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIgpzdW1tYXJpc2VfcHJlZGljdGlvbnMucHkgLS0gc2xpY2UgaW5kaWFuX3ByZWRpY3Rpb25zLmNzdiBieSBkb2N1bWVudCAvIGV4Y2x1c2lvbnMuCgogICAgcHl0aG9uIHN1bW1hcmlzZV9wcmVkaWN0aW9ucy5weSBpbmRpYW5fcHJlZGljdGlvbnMuY3N2CiAgICBweXRob24gc3VtbWFyaXNlX3ByZWRpY3Rpb25zLnB5IGluZGlhbl9wcmVkaWN0aW9ucy5jc3YgLS1leGNsdWRlIG5vaWRhIC0tZXhjbHVkZV9vdGhlcgoKU2NvcmVzIG9ubHkgY2F0ZWdvcmllcyB0aGUgbW9kZWwgd2FzIHRyYWluZWQgb24uIC0tZXhjbHVkZSBkcm9wcyBkb2N1bWVudHMgd2hvc2UgaWQKY29udGFpbnMgdGhlIGdpdmVuIHRleHQgKGUuZy4gT0NSLW5vaXN5IG9uZXMpLiAtLWV4Y2x1ZGVfb3RoZXIgZHJvcHMgdHJ1ZSAnT3RoZXInLgpJbnRlcnZhbHMgYXJlIFdpbHNvbiA5NSU7IHdpdGggYSBmZXcgZG96ZW4gY2xhdXNlcyB0aGV5IGFyZSB3aWRlLgoiIiIKaW1wb3J0IGFyZ3BhcnNlLCBjc3YsIG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKCgpkZWYgd2lsc29uKGssIG4sIHo9MS45Nik6CiAgICBpZiBuID09IDA6CiAgICAgICAgcmV0dXJuIChmbG9hdCgibmFuIiksKSAqIDIKICAgIHAgPSBrIC8gbjsgZCA9IDEgKyB6ICogeiAvIG47IGMgPSAocCArIHogKiB6IC8gKDIgKiBuKSkgLyBkCiAgICBoID0geiAqIG1hdGguc3FydChwICogKDEgLSBwKSAvIG4gKyB6ICogeiAvICg0ICogbiAqIG4pKSAvIGQKICAgIHJldHVybiBtYXgoMCwgYyAtIGgpLCBtaW4oMSwgYyArIGgpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCJjc3ZfcGF0aCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXhjbHVkZSIsIGFjdGlvbj0iYXBwZW5kIiwgZGVmYXVsdD1bXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leGNsdWRlX290aGVyIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKICAgIHJvd3MgPSBsaXN0KGNzdi5EaWN0UmVhZGVyKG9wZW4oYS5jc3ZfcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikpKQogICAgcm93cyA9IFtyIGZvciByIGluIHJvd3MgaWYgclsic2Vlbl9jYXRlZ29yeSJdID09ICIxIgogICAgICAgICAgICBhbmQgbm90IGFueSh4Lmxvd2VyKCkgaW4gclsiZG9jdW1lbnQiXS5sb3dlcigpIGZvciB4IGluIGEuZXhjbHVkZSkKICAgICAgICAgICAgYW5kIG5vdCAoYS5leGNsdWRlX290aGVyIGFuZCByWyJ0cnVlX2NhdGVnb3J5Il0gPT0gIk90aGVyIildCiAgICBkZWYgbGluZShuYW1lLCBycyk6CiAgICAgICAgayA9IHN1bShpbnQoclsiY29ycmVjdCJdKSBmb3IgciBpbiBycyk7IG4gPSBsZW4ocnMpOyBsbywgaGkgPSB3aWxzb24oaywgbikKICAgICAgICBwcmludChmInwge25hbWV9IHwge259IHwge2svbjouM2Z9IHwge2xvOi4zZn0gdG8ge2hpOi4zZn0gfCIgaWYgbiBlbHNlIGYifCB7bmFtZX0gfCAwIHwgbi9hIHwgfCIpCiAgICBwcmludCgifCBTbGljZSB8IG4gfCBhY2N1cmFjeSB8IDk1JSBpbnRlcnZhbCB8XG58LS0tfC0tLXwtLS18LS0tfCIpCiAgICBsaW5lKCJBTEwgKHNlZW4gY2F0ZWdvcmllcykiLCByb3dzKQogICAgYnkgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIHIgaW4gcm93czogYnlbclsiZG9jdW1lbnQiXV0uYXBwZW5kKHIpCiAgICBmb3IgZCwgcnMgaW4gc29ydGVkKGJ5Lml0ZW1zKCksIGtleT1sYW1iZGEgeDogLWxlbih4WzFdKSk6IGxpbmUoZCwgcnMpCiAgICBwcmludCgiXG58IFRydWUgY2F0ZWdvcnkgfCBuIHwgYWNjdXJhY3kgfCBtb3N0IGNvbW1vbiB3cm9uZyBwcmVkaWN0aW9uIHxcbnwtLS18LS0tfC0tLXwtLS18IikKICAgIGJjID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciByIGluIHJvd3M6IGJjW3JbInRydWVfY2F0ZWdvcnkiXV0uYXBwZW5kKHIpCiAgICBmb3IgYywgcnMgaW4gc29ydGVkKGJjLml0ZW1zKCksIGtleT1sYW1iZGEgeDogLWxlbih4WzFdKSk6CiAgICAgICAgd3JvbmcgPSBkZWZhdWx0ZGljdChpbnQpCiAgICAgICAgZm9yIHIgaW4gcnM6CiAgICAgICAgICAgIGlmIG5vdCBpbnQoclsiY29ycmVjdCJdKTogd3JvbmdbclsicHJlZF9jYXRlZ29yeSJdXSArPSAxCiAgICAgICAgdG9wID0gbWF4KHdyb25nLCBrZXk9d3JvbmcuZ2V0KSBpZiB3cm9uZyBlbHNlICItIgogICAgICAgIHByaW50KGYifCB7Y30gfCB7bGVuKHJzKX0gfCB7c3VtKGludChyWydjb3JyZWN0J10pIGZvciByIGluIHJzKS9sZW4ocnMpOi4yZn0gfCB7dG9wfSB8IikKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"}.items():
    open(f"{WORK}/{name}", "wb").write(base64.b64decode(b))
print("ready:", os.listdir(WORK))

In [ ]:
os.chdir(WORK)
for k in FOLDS:
    out = f"{WORK}/built_fold{k}"
    shutil.copytree(f"{WORK}/built_v2", out, dirs_exist_ok=True)
    cmd = [sys.executable, "eval_indian.py", "--run_dir", WORK, "--fold", str(k),
           "--ckpt", f"{MODEL_SRC}/ckpt_joint_cls_span_fold{k}.pt", "--indian_dir", out,
           "--mapped_jsonl", f"{MODEL_SRC}/clauses_mapped.jsonl", "--cuad_json", f"{MODEL_SRC}/CUAD_v1.json",
           "--rules", f"{WORK}/indian_legal_rules.csv", "--clause_batch", "2", "--span_batch", "1"]
    print("=== fold", k, "==="); r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-1500:] if r.returncode else "")

In [ ]:
for k in FOLDS:
    print(f"\n##### fold {k}: ALL docs"); get = f"{WORK}/built_fold{k}/indian_predictions.csv"
    print(subprocess.run([sys.executable, "summarise_predictions.py", get], capture_output=True, text=True).stdout)
    print(f"##### fold {k}: excluding Noida + true Other")
    print(subprocess.run([sys.executable, "summarise_predictions.py", get, "--exclude", "noida", "--exclude_other"], capture_output=True, text=True).stdout)

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/indian_eval_results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in FOLDS:
        for f in ["indian_eval.json", "indian_eval.md", "indian_predictions.csv", "indian_rule_flags.csv"]:
            p = f"{WORK}/built_fold{k}/{f}"
            if os.path.exists(p): z.write(p, f"fold{k}/{f}")
print("download /kaggle/working/indian_eval_results_v2.zip from the Output tab")